# COVID-19 Case Classification Using Machine Learning

## Project Overview
This project analyzes a global COVID-19 dataset and builds machine-learning classification models to identify countries with **high total COVID-19 cases**.

### Workflow
1. Load and inspect the dataset
2. Clean numeric columns
3. Remove summary rows
4. Analyze missing values
5. Perform exploratory data analysis
6. Create a binary target
7. Split data into training and testing sets
8. Preprocess features using an imputation pipeline
9. Train Logistic Regression, Decision Tree and Random Forest models
10. Evaluate models using Accuracy, Precision, Recall and F1-score
11. Visualize the confusion matrix
12. Apply PCA for 2D visualization

> **Note:** The target is created from `Total Cases`, so `Total Cases` is excluded from the input features to avoid direct target leakage.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings("ignore")


## 1. Load Dataset

In [ ]:
# Keep the CSV file in the same folder as this notebook
df = pd.read_csv("COVID-19 Global - Dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())


## 2. Understand the Data

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())


## 3. Data Cleaning

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Remove commas and convert numeric columns
for col in df.columns:
    if col != "Country":
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .replace("nan", np.nan)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove summary/aggregate rows such as "Total:"
df = df[~df["Country"].astype(str).str.startswith("Total")].copy()

print("Shape after cleaning:", df.shape)
display(df.head())


## 4. Missing-Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)

print(missing)

plt.figure(figsize=(10, 5))
missing[missing > 0].plot(kind="bar")
plt.title("Missing Values by Feature")
plt.xlabel("Features")
plt.ylabel("Number of Missing Values")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## 5. Exploratory Data Analysis

In [ ]:
# Top 10 countries by total cases
top10 = df.nlargest(10, "Total Cases")

plt.figure(figsize=(10, 5))
plt.bar(top10["Country"], top10["Total Cases"])
plt.title("Top 10 Countries by Total COVID-19 Cases")
plt.xlabel("Country")
plt.ylabel("Total Cases")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of total cases
plt.figure(figsize=(8, 5))
plt.hist(df["Total Cases"].dropna(), bins=30)
plt.title("Distribution of Total COVID-19 Cases")
plt.xlabel("Total Cases")
plt.ylabel("Number of Countries")
plt.tight_layout()
plt.show()


## 6. Create Target Variable

In [ ]:
# Use the median total cases as the threshold
threshold = df["Total Cases"].median()

# 1 = High Total Cases, 0 = Low Total Cases
df["Target"] = (df["Total Cases"] > threshold).astype(int)

print("Median threshold:", threshold)
print("\nTarget distribution:")
print(df["Target"].value_counts())


### Target Definition
- **1:** Country has total cases above the dataset median
- **0:** Country has total cases at or below the dataset median

`Total Cases` is removed from the model features because it was used to create the target.


## 7. Prepare Features and Target

In [ ]:
# Keep only numeric predictors
X = df.drop(columns=["Country", "Total Cases", "Target"])
y = df["Target"]

print("Features used:")
print(X.columns.tolist())
print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)


## 8. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 9. Build Machine-Learning Models

In [ ]:
# Preprocessing: median imputation + standardization
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        max_depth=8
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)
results_df


## 10. Model Comparison

In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]

results_df.set_index("Model")[metrics].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Machine-Learning Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 11. Classification Report

In [ ]:
for name, pipeline in trained_models.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    y_pred = pipeline.predict(X_test)
    print(classification_report(y_test, y_pred, zero_division=0))


## 12. Confusion Matrix

In [ ]:
# Display the confusion matrix for the Random Forest model
best_model_name = "Random Forest"
best_model = trained_models[best_model_name]

ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test
)

plt.title("Confusion Matrix - Random Forest")
plt.show()


## 13. Random Forest Feature Importance

In [ ]:
# The Random Forest is inside the pipeline
rf_model = trained_models["Random Forest"].named_steps["model"]

# Median-imputed feature values are used by the model
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance)

plt.figure(figsize=(10, 5))
feature_importance.plot(kind="bar")
plt.title("Random Forest Feature Importance")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## 14. PCA Visualization

In [ ]:
# Impute and standardize the complete feature set for visualization
X_processed = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]).fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y,
    alpha=0.8
)
plt.title("PCA Visualization of COVID-19 Data")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label="Target")
plt.tight_layout()
plt.show()


## 15. Conclusion

This project demonstrates an end-to-end machine-learning workflow on COVID-19 country-level data. The work includes data cleaning, missing-value handling, exploratory analysis, target creation, train-test splitting, preprocessing, classification using Logistic Regression, Decision Tree and Random Forest, model evaluation, feature-importance analysis and PCA visualization.

The project can be extended by using a larger time-series COVID-19 dataset, adding more meaningful prediction targets, performing hyperparameter tuning, and validating models with cross-validation.
